# Ансамбли и out-of-fold предсказания

Авторы BelOAI. CC BY 4.0.

Выполняйте ячейки сверху вниз. Сначала решите задачи, затем обратитесь к решениям.

## Чему научимся

Строить OOF-прогнозы и объяснять, как обучать модель, объединяющую прогнозы других моделей.

## Прогноз для незнакомого объекта

OOF, out-of-fold, — прогноз обучающей строки моделью, которая не училась на этой строке. Разделим данные разработки на K блоков: для каждого обучим модель на остальных и запишем прогноз только в оставленный блок. За один полный цикл перекрёстной проверки каждая строка получает один OOF-прогноз.

Такие прогнозы позволяют оценивать качество и обучать метамодель — правило, объединяющее выходы нескольких базовых моделей. Если вместо OOF использовать прогнозы базовых моделей на их собственном train, метамодель увидит слишком лёгкие сигналы, отличающиеся от реальных новых объектов.

## Разобранный пример: среднее двух моделей

Первый прогноз 8, второй 12. Простое среднее 10 не требует обучения. Взвешенная сумма 0.25·8+0.75·12=11 уже выражает предпочтение второй модели. Если веса выбирают по качеству, это тоже обучение. Чтобы его проверить, нужны данные, по которым эти веса не выбирали.

Чтобы получить прогноз для итогового теста, базовые модели обучают на всех разрешённых данных разработки. Другой вариант — усреднить прогнозы моделей из отдельных разбиений. Способ выбирают заранее. При применении метамодели сохраняют тот же порядок столбцов с прогнозами. Нельзя менять смысл признака «прогноз модели А» после её обучения.

## Остаточная осторожность

OOF не защищает от групповых утечек и подготовки, обученной на всей таблице. Группы и время должны учитываться и в разбиении OOF. Если метамодель подбирается по тем же OOF много раз, для итоговой оценки всё равно остаётся отдельный тест.

В ноутбуке cross_val_predict строит OOF-прогнозы всего Pipeline; KFold с явным seed задаёт разбиение. Проверяем форму и отсутствие пересечения train/validation в каждом шаге.

## Разобранный пример: проверяем вычисление

Все числа в примере придуманы для этого урока. Выполните код, затем измените вход и объясните новый результат.

In [ ]:
import numpy as np
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
X = np.arange(12, dtype=float).reshape(-1, 1)
y = 2 * X[:, 0] + 1
cv = KFold(n_splits=3, shuffle=True, random_state=17)
for train, valid in cv.split(X):
    assert set(train).isdisjoint(set(valid))
model = Pipeline([("scale", StandardScaler()), ("ridge", Ridge(alpha=1.))])
oof = cross_val_predict(model, X, y, cv=cv)
assert oof.shape == y.shape
assert np.isfinite(oof).all()
print(oof)


## Типичные ошибки

Не нарушайте порядок строк OOF. Не обучайте подготовку глобально перед cross_val_predict.

## Задачи

### 1. Базовая

Что означает OOF для одной строки?

<details>
<summary>Подсказка</summary>

Она исключена из обучения её модели.

</details>



### 2. Базовая

Нужно ли обучать простое среднее двух прогнозов?

<details>
<summary>Подсказка</summary>

Веса уже фиксированы.

</details>



### 3. Прикладная

Найдите среднее 6 и 10.

<details>
<summary>Подсказка</summary>

Сложите и разделите на два.

</details>



### 4. Прикладная

Найдите 0.2·6+0.8·10.

<details>
<summary>Подсказка</summary>

Взвешенные вклады.

</details>



### 5. Олимпиадная

Почему train-прогнозы опасны для метамодели?

<details>
<summary>Подсказка</summary>

Базовые модели уже видели ответы этих объектов.

</details>



### 6. Олимпиадная

Гарантирует ли OOF отсутствие дубликатов по разные стороны?

<details>
<summary>Подсказка</summary>

Он следует заданному разбиению.

</details>



## Итоги и самопроверка

Нарисуйте получение одного OOF-столбца и финального тестового столбца. Где учатся веса объединения?

Для перехода дальше достаточно базовых и прикладных задач. Олимпиадные задачи можно отложить.


## Решения задач

### Задача 1

Прогноз получен без обучения базовой модели на этой строке.

### Задача 2

Нет. Но выбор этих весов по данным был бы подбором.

### Задача 3

8.

### Задача 4

1.2+8=9.2.

### Задача 5

Их ошибки слишком оптимистичны, а признаки для метамодели не соответствуют новым объектам.

### Задача 6

Нет. Нужна схема разбиения с учётом групп или времени. Преобразования нужно обучать заново на обучающей части каждого разбиения.